# Étape 4 — v2 : compétences en dernier + réparation des boucles

Changements par rapport à la v1 :
1. **`competences` est le dernier champ** du JSON (schéma, prompt et exemples d'entraînement). Si le modèle boucle sur la liste, tous les autres champs sont déjà écrits.
2. **Réparation** : si la sortie n'est pas du JSON valide, on coupe la liste au premier doublon et on referme le JSON. On mesure **brut** et **réparé** séparément.

Même découpage train/test que la v1 (`test.jsonl` réutilisé), mêmes hyperparamètres : la seule différence est l'ordre des champs, pour que la comparaison soit juste.

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU T4. Durée : 35 à 50 min.

In [ ]:
# 1. Installation + Google Drive
!pip install -q transformers accelerate peft bitsandbytes datasets pydantic pandas

from google.colab import drive
drive.mount('/content/drive')

import os
DOSSIER = '/content/drive/MyDrive/job-offer-extractor/data'
RESULTATS = '/content/drive/MyDrive/job-offer-extractor/resultats'
TRAIN = f'{DOSSIER}/train.jsonl'
TEST = f'{DOSSIER}/test.jsonl'
assert os.path.exists(TEST), 'test.jsonl introuvable : lance d’abord l’étape 3'

In [ ]:
# 2. Connexion Hugging Face
from huggingface_hub import login
login()

In [ ]:
# 3. Schéma avec `competences` EN DERNIER
import json, re
from typing import Literal, Optional
from pydantic import BaseModel, ValidationError

class Offre(BaseModel):
    poste: str
    entreprise: Optional[str]
    ville: Optional[str]
    type_contrat: Optional[Literal['PFE', 'stage', 'CDI', 'CDD', 'SIVP', 'freelance']]
    mode_travail: Optional[Literal['présentiel', 'hybride', 'remote']]
    duree_mois: Optional[float]
    remuneration: Optional[str]
    langue_offre: Literal['fr', 'en', 'ar']
    competences: list[str]

ORDRE = list(Offre.model_fields)
SCHEMA = json.dumps(Offre.model_json_schema(), ensure_ascii=False)

REGLES = '''Règles :
- type_contrat = "PFE" si l'offre parle de PFE ou de fin d'études, sinon "stage" pour un stage. Si plusieurs types sont cités, prendre le premier.
- remuneration = le montant s'il est donné ; "rémunéré" si l'offre dit seulement rémunéré ; une prime n'est pas un salaire ; "Confidentiel" = null.
- competences = uniquement des compétences techniques ou métier (outils, langages, domaines). Jamais d'années d'expérience, de diplôme, de tâches ni de qualités humaines. Chaque compétence apparaît une seule fois.
- langue_offre = UNE seule valeur, la langue dominante.
- ville et entreprise sont deux champs séparés.
- Si une information est absente, mets null. Ne devine jamais.'''

SYSTEM_PROMPT = (
    "Tu extrais les informations d'une offre d'emploi ou de stage. "
    'Réponds UNIQUEMENT avec un JSON valide respectant ce schéma :\n'
    + SCHEMA + '\n\n' + REGLES
)
print('Ordre des champs :', ORDRE)

In [ ]:
# 4. Charger le découpage existant (le même que la v1)
from collections import Counter

def lire(chemin):
    with open(chemin, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train, test = lire(TRAIN), lire(TEST)
print(f'Train : {len(train)} | Test : {len(test)}')
print('Test, langues :', dict(Counter(d['extraction']['langue_offre'] for d in test)))

In [ ]:
# 5. Charger le modèle de base en 4 bits
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')

In [ ]:
# 6. Exemples d'entraînement avec les champs dans le nouvel ordre
from datasets import Dataset

MAX_LEN = 2560

def construire(ex):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': ex['offre']}]
    prompt_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True
    )['input_ids']
    extraction = {k: ex['extraction'][k] for k in ORDRE}
    reponse = json.dumps(extraction, ensure_ascii=False) + tokenizer.eos_token
    reponse_ids = tokenizer(reponse, add_special_tokens=False)['input_ids']
    input_ids = (prompt_ids + reponse_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + reponse_ids)[:MAX_LEN]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids), 'labels': labels}

train_ds = Dataset.from_list([construire(ex) for ex in train])
print(len(train_ds), 'exemples | longueur max', max(len(x) for x in train_ds['input_ids']))
print('Exemple de cible :', json.dumps({k: train[0]['extraction'][k] for k in ORDRE}, ensure_ascii=False)[:200], '…')

In [ ]:
# 7. LoRA + entraînement (mêmes hyperparamètres que la v1)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
))

args = TrainingArguments(
    output_dir='/content/qlora-offres-v2',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=2,
    fp16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    seed=42,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
model.config.use_cache = False
trainer.train()
model.config.use_cache = True

In [ ]:
# 8. Sauvegarder tout de suite (avant l'évaluation, au cas où Colab se déconnecte)
ADAPTER = '/content/drive/MyDrive/job-offer-extractor/adapter_v2'
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print('Adaptateur v2 sauvegardé dans', ADAPTER)

## Évaluation : génération une seule fois, puis scores brut et réparé

On génère les 14 sorties une fois (environ 12 min), puis on calcule deux scores sur **les mêmes sorties** :
- **brut** : la sortie telle quelle ;
- **réparé** : si le JSON est invalide, la liste des compétences est coupée au premier doublon et le JSON est refermé.

In [ ]:
# 9. Génération, réparation et scores
import pandas as pd

def nettoyer(texte):
    return texte.strip().removeprefix('```json').removesuffix('```').strip()

@torch.no_grad()
def extraire(offre, max_new_tokens=900):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': offre}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def reparer(brut):
    """Coupe la liste des compétences au premier doublon et referme le JSON.
    Fonctionne parce que `competences` est le dernier champ."""
    t = nettoyer(brut)
    i = t.find('"competences"')
    if i < 0:
        return t
    debut = t.index('[', i)
    items, vus = [], set()
    for m in re.finditer(r'"((?:[^"\\]|\\.)*)"', t[debut:]):
        try:
            item = json.loads('"' + m.group(1) + '"')
        except json.JSONDecodeError:
            break
        if item.strip().lower() in vus:
            break
        vus.add(item.strip().lower())
        items.append(item)
    return t[:debut] + json.dumps(items, ensure_ascii=False) + '}'

CHAMPS = ['poste', 'entreprise', 'ville', 'type_contrat', 'mode_travail',
          'duree_mois', 'remuneration', 'langue_offre']

def norm(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return round(float(v), 2)
    return ' '.join(str(v).lower().split())

def egal(champ, pred, vrai):
    if champ == 'duree_mois' and pred is not None and vrai is not None:
        try:
            return abs(float(pred) - float(vrai)) < 0.05
        except (TypeError, ValueError):
            return False
    return norm(pred) == norm(vrai)

def scorer(nom, sorties, reparation):
    lignes, tp, fp, fn = [], 0, 0, 0
    for ex, brut in zip(test, sorties):
        vrai = ex['extraction']
        ligne = {'url': ex.get('url'), 'langue': vrai['langue_offre'], 'sortie_brute': brut,
                 'repare': False}
        try:
            pred = json.loads(nettoyer(brut))
            ligne['json_valide'] = True
        except json.JSONDecodeError:
            ligne['json_valide'] = False
            pred = {}
            if reparation:
                try:
                    pred = json.loads(reparer(brut))
                    ligne['repare'] = True
                except (json.JSONDecodeError, ValueError):
                    pred = {}
        try:
            Offre.model_validate(pred)
            ligne['schema_ok'] = True
        except ValidationError:
            ligne['schema_ok'] = False
        for c in CHAMPS:
            ligne[c] = egal(c, pred.get(c), vrai.get(c))
        p = {norm(x) for x in (pred.get('competences') or []) if isinstance(x, str)}
        v = {norm(x) for x in vrai['competences']}
        tp += len(p & v); fp += len(p - v); fn += len(v - p)
        lignes.append(ligne)
    df = pd.DataFrame(lignes)
    prec = tp / (tp + fp) if tp + fp else 0
    rapp = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * prec * rapp / (prec + rapp) if prec + rapp else 0
    resume = {'JSON valide': df['json_valide'].mean(),
              'Réparées': df['repare'].mean(),
              'Schéma respecté': df['schema_ok'].mean()}
    resume.update({c: df[c].mean() for c in CHAMPS})
    resume.update({'Compétences précision': prec, 'Compétences rappel': rapp, 'Compétences F1': f1})
    df.to_csv(f'{RESULTATS}/details_{nom}.csv', index=False)
    return pd.Series(resume, name=nom).round(3)

model.eval()
sorties = []
for i, ex in enumerate(test):
    sorties.append(extraire(ex['offre']))
    print(f'{i+1}/{len(test)}', end='\r')

res_v2_brut = scorer('finetune_v2_brut', sorties, reparation=False)
res_v2_repare = scorer('finetune_v2_repare', sorties, reparation=True)

In [ ]:
# 10. Tableau final : base, v1, v2 brut, v2 réparé
anciens = pd.read_csv(f'{RESULTATS}/comparaison_v1_decodage.csv', index_col=0)[['base', 'finetune_v1']]
final = pd.concat([anciens, res_v2_brut, res_v2_repare], axis=1)
final.to_csv(f'{RESULTATS}/comparaison_finale.csv')
print(f'Test : {len(test)} offres (1 offre = {100/len(test):.1f} points)')
final

In [ ]:
# 11. Les sorties qui ont eu besoin d'une réparation
d = pd.read_csv(f'{RESULTATS}/details_finetune_v2_repare.csv')
for _, r in d[d['repare']].iterrows():
    print('URL :', r['url'])
    print('Réparé :', reparer(r['sortie_brute'])[:400])
    print('-' * 60)
if not d['repare'].any():
    print('Aucune réparation nécessaire : la v2 ne boucle plus sur le test.')

In [ ]:
# 12. Pousser l'adaptateur v2 sur Hugging Face (privé)
REPO = 'mayss1/qwen2.5-1.5b-offres-tn-lora'
model.push_to_hub(REPO, private=True, commit_message='v2 : competences en dernier')
tokenizer.push_to_hub(REPO, private=True)
print('v2 poussée sur', REPO)

In [ ]:
d = pd.read_csv(f'{RESULTATS}/details_finetune_v2_brut.csv')
for ex, (_, r) in zip(test, d.iterrows()):
    pred = json.loads(nettoyer(r['sortie_brute']))
    vrai = ex['extraction']
    if not r['poste']:
        print('POSTE  attendu :', vrai['poste'], '| prédit :', pred.get('poste'))
    p = {norm(x) for x in pred.get('competences', [])}
    v = {norm(x) for x in vrai['competences']}
    if p != v:
        print('  inventées :', sorted(p - v)[:8])
        print('  oubliées  :', sorted(v - p)[:8])
    print('-' * 50)


In [ ]:
def present(terme, texte):
    return terme.lower() in texte.lower()

total_inv, inv_dans_texte, total_gold, gold_absent = 0, 0, 0, 0
for ex, (_, r) in zip(test, d.iterrows()):
    pred = json.loads(nettoyer(r['sortie_brute']))
    texte = ex['offre']
    p = {norm(x) for x in pred.get('competences', [])}
    v = {norm(x) for x in ex['extraction']['competences']}
    for t in p - v:
        total_inv += 1
        if present(t, texte):
            inv_dans_texte += 1
            print('Présent dans l’offre mais pas annoté :', t)
    for t in v:
        total_gold += 1
        gold_absent += not present(t, texte)

print(f'\n« Inventées » réellement présentes dans le texte : {inv_dans_texte}/{total_inv}')
print(f'Étiquettes absentes mot pour mot du texte : {gold_absent}/{total_gold}')

In [ ]:
# Réannotation v3 — application (poste + competences uniquement), journal, sauvegarde
import shutil, csv, copy
E[2][1] = [("Système d’Information Géographique" if x == "Système d'Information Géographique" else x) for x in E[2][1]]

ARABE = re.compile(r'[؀-ۿ]')
def raison_renommage(avant, apres):
    if isinstance(apres, list): return "séparation d'une compétence composée"
    if ARABE.search(avant): return 'nom canonique en alphabet latin'
    if re.search(r'(?<!\w)(IA|AI)(?!\w)|intelligence artificielle', avant, re.I): return 'règle IA → « intelligence artificielle »'
    if avant.lower() == apres.lower(): return 'harmonisation de la graphie (casse)'
    return 'nom canonique / graphie officielle (harmonisée sur tout le jeu)'

# 1) Sauvegarde des fichiers v2
SAUV = f'{DOSSIER}/sauvegarde_avant_v3'
os.makedirs(SAUV, exist_ok=True)
for f in ['offres_propres.jsonl', 'train.jsonl', 'test.jsonl']:
    if not os.path.exists(f'{SAUV}/{f}'):
        shutil.copy(f'{DOSSIER}/{f}', f'{SAUV}/{f}')

# 2) Application + journal
journal, nouv = [], {}
avant_moy = sum(len(o['extraction']['competences']) for o in propres) / len(propres)
propres_v3 = copy.deepcopy(propres)
for i, o in enumerate(propres_v3):
    poste_n, final, ren, rm = E[i]
    ext, url, texte = o['extraction'], o['url'], o['offre']
    old = list(ext['competences'])
    if poste_n and poste_n != ext['poste']:
        r = 'retrait de la mention H/F' if re.search(r'\(?\b(H/F|F/H)\b\)?', ext['poste']) else 'titre recopié depuis l’offre (le libellé précédent ne correspondait pas au titre)'
        journal.append({'url': url, 'champ': 'poste', 'avant': ext['poste'], 'après': poste_n, 'raison': r})
        ext['poste'] = poste_n
    for k, why in rm.items():
        journal.append({'url': url, 'champ': 'competences', 'avant': k, 'après': '', 'raison': 'retrait : ' + why})
    for k, v in ren.items():
        journal.append({'url': url, 'champ': 'competences', 'avant': k, 'après': ' | '.join(v) if isinstance(v, list) else v, 'raison': 'renommage : ' + raison_renommage(k, v)})
    issus = {x for v in ren.values() for x in (v if isinstance(v, list) else [v])}
    for x in final:
        if x not in old and x not in issus:
            why = 'présente dans le texte, passe le test « rubrique Compétences d’un CV »' if present_mot(x, texte) else 'nom canonique d’une compétence citée dans le texte sous une autre forme'
            journal.append({'url': url, 'champ': 'competences', 'avant': '', 'après': x, 'raison': 'ajout : ' + why})
    ext['competences'] = final
    Offre.model_validate(ext)
    nouv[url] = o

# 3) Écriture : mêmes offres, même ordre, même découpage (identifié par url)
def ecrire(chemin, lignes):
    with open(chemin, 'w', encoding='utf-8') as f:
        for l in lignes: f.write(json.dumps(l, ensure_ascii=False) + '\n')
def maj(liste):
    out = []
    for o in liste:
        n = copy.deepcopy(o); n['extraction']['poste'] = nouv[o['url']]['extraction']['poste']
        n['extraction']['competences'] = nouv[o['url']]['extraction']['competences']
        out.append(n)
    return out
tr_v3, te_v3 = maj(tr), maj(te)
ecrire(f'{DOSSIER}/offres_propres.jsonl', propres_v3)
ecrire(f'{DOSSIER}/train.jsonl', tr_v3)
ecrire(f'{DOSSIER}/test.jsonl', te_v3)
pd.DataFrame(journal, columns=['url', 'champ', 'avant', 'après', 'raison']).to_csv(f'{DOSSIER}/journal_reannotation_v3.csv', index=False, encoding='utf-8')

# 4) Vérifications après écriture
p2, tr2, te2 = (lire_jsonl(f'{DOSSIER}/{f}') for f in ['offres_propres.jsonl', 'train.jsonl', 'test.jsonl'])
assert [o['url'] for o in te2] == [o['url'] for o in te] and len(te2) == 14
assert [o['url'] for o in tr2] == [o['url'] for o in tr] and len(tr2) == 54
CH = ['entreprise', 'ville', 'type_contrat', 'mode_travail', 'duree_mois', 'remuneration', 'langue_offre']
for a, b in zip(propres + tr + te, p2 + tr2 + te2):
    assert a['offre'] == b['offre'] and all(a['extraction'][c] == b['extraction'][c] for c in CH)
    assert {k: v for k, v in a.items() if k != 'extraction'} == {k: v for k, v in b.items() if k != 'extraction'}
    Offre.model_validate(b['extraction'])
assert all(nouv[o['url']]['extraction'] == o['extraction'] for o in tr2 + te2)
J = pd.DataFrame(journal)
c = J[J.champ == 'competences']
print('Ajouts :', (c.raison.str.startswith('ajout')).sum(), '| Retraits :', (c.raison.str.startswith('retrait')).sum(), '| Renommages :', (c.raison.str.startswith('renommage')).sum(), '| Postes modifiés :', (J.champ == 'poste').sum())
print('Compétences / offre : avant %.2f → après %.2f' % (avant_moy, sum(len(o['extraction']['competences']) for o in p2) / len(p2)))
print('Test : avant %.2f → après %.2f' % (sum(len(o['extraction']['competences']) for o in te)/14, sum(len(o['extraction']['competences']) for o in te2)/14))
print(J[J.champ == 'poste'][['avant', 'après']].to_string())
print(c.raison.str.split(' : ').str[1].value_counts().to_string())

In [ ]:
# Rescoring v3 SANS régénération : sorties brutes enregistrées vs NOUVELLES étiquettes du test
json.dump({str(k): v for k, v in E.items()}, open(f'{DOSSIER}/etiquettes_v3.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
test = lire_jsonl(f'{DOSSIER}/test.jsonl')   # nouvelles étiquettes, même ordre

def norm_souple(s, poste=False):
    s = str(s).lower()
    if poste: s = re.sub(r'\(?\s*\b(h/f|f/h)\b\s*\)?', ' ', s)
    s = re.sub(r'[^\w\s#+.]', ' ', s)          # ponctuation retirée (on garde # + . : C, C#, C++ et .NET restent distincts)
    s = re.sub(r'(?<!\w)\.|\.(?!\w)', ' ', s)    # points isolés retirés
    return ' '.join(s.split())

def comp_souples(liste):
    out = set()
    for x in liste:
        for part in str(x).split('/'):
            n = norm_souple(part)
            if n: out.add(n)
    return out

def parse(brut):
    try:
        p = json.loads(nettoyer(str(brut)))
        return (p if isinstance(p, dict) else {}), True
    except json.JSONDecodeError:
        return {}, False

def rescorer(nom):
    d = pd.read_csv(f'{RESULTATS}/details_{nom}.csv')
    assert list(d['url']) == [ex['url'] for ex in test], f'ordre différent pour {nom}'
    lignes, s, sv, anc = [], [0, 0, 0], [0, 0, 0], [0, 0]
    for ex, brut in zip(test, d['sortie_brute']):
        vrai, (pred, ok) = ex['extraction'], parse(brut)
        l = {'json_valide': ok}
        try: Offre.model_validate(pred); l['schema_ok'] = True
        except ValidationError: l['schema_ok'] = False
        for c in CHAMPS: l[c] = egal(c, pred.get(c), vrai.get(c))
        l['poste_souple'] = pred.get('poste') is not None and norm_souple(pred.get('poste'), True) == norm_souple(vrai['poste'], True)
        comps = [x for x in (pred.get('competences') or []) if isinstance(x, str)] if isinstance(pred.get('competences'), list) else []
        p, v = {norm(x) for x in comps}, {norm(x) for x in vrai['competences']}
        s[0] += len(p & v); s[1] += len(p - v); s[2] += len(v - p)
        ps, vs = comp_souples(comps), comp_souples(vrai['competences'])
        sv[0] += len(ps & vs); sv[1] += len(ps - vs); sv[2] += len(vs - ps)
        uniq = {x.strip().lower(): x for x in comps if x.strip()}
        anc[0] += sum(present_mot(x, ex['offre']) for x in uniq.values()); anc[1] += len(uniq)
        lignes.append(l)
    df = pd.DataFrame(lignes)
    def prf(tp, fp, fn):
        pr = tp / (tp + fp) if tp + fp else 0; rc = tp / (tp + fn) if tp + fn else 0
        return pr, rc, (2 * pr * rc / (pr + rc) if pr + rc else 0)
    r = {'JSON valide': df.json_valide.mean(), 'Schéma respecté': df.schema_ok.mean()}
    r.update({c: df[c].mean() for c in CHAMPS})
    r['poste_souple'] = df.poste_souple.mean()
    r.update(dict(zip(['Compétences précision', 'Compétences rappel', 'Compétences F1'], prf(*s))))
    r.update(dict(zip(['Compétences souples précision', 'Compétences souples rappel', 'Compétences souples F1'], prf(*sv))))
    r['Taux d’ancrage (mot entier)'] = anc[0] / anc[1] if anc[1] else 0
    r['Nb compétences prédites'] = anc[1]
    return pd.Series(r, name=nom)

v3 = pd.concat([rescorer(n) for n in ['base', 'finetune_v1', 'finetune_v2_brut']], axis=1)
v3.loc['Nb compétences prédites'] = v3.loc['Nb compétences prédites'].astype(int)
v3 = v3.round(3)
v3.to_csv(f'{RESULTATS}/comparaison_v3_rescoring.csv')
ancien = pd.read_csv(f'{RESULTATS}/comparaison_finale.csv', index_col=0)
print('Étiquettes compétences du test :', sum(len(e['extraction']['competences']) for e in test))
print(v3.to_string())
print('\nAnciennes étiquettes (comparaison_finale.csv), pour mémoire :')
print(ancien.loc[[i for i in ['poste', 'Compétences précision', 'Compétences rappel', 'Compétences F1'] if i in ancien.index]].to_string())

In [ ]:
# GUIDE_ANNOTATION.md : section « Compétences » (créée, ou remplacée si elle existe déjà)
import base64
SECTION = base64.b64decode('IyMgQ29tcMOpdGVuY2VzCgpDaGFtcCBgY29tcGV0ZW5jZXNgIGR1IHNjaMOpbWEgYE9mZnJlYCAoZGVybmllciBjaGFtcCBkdSBKU09OKS4gUsOoZ2xlcyBhcHBsaXF1w6llcyBsb3JzIGRlIGxhIHLDqWFubm90YXRpb24gdjMgKGpvdXJuYWwgOiBgZGF0YS9qb3VybmFsX3JlYW5ub3RhdGlvbl92My5jc3ZgLCDDqXRpcXVldHRlcyA6IGBkYXRhL2V0aXF1ZXR0ZXNfdjMuanNvbmApLgoKIyMjIFRlc3QgZGUgZMOpY2lzaW9uCgo+ICoqRXN0LWNlIHF1ZSBjZWxhIHMnw6ljcmlyYWl0IGRhbnMgbGEgcnVicmlxdWUgwqsgQ29tcMOpdGVuY2VzIMK7IGQndW4gQ1YgPyoqCgoqKk9VSSoqIDogbGFuZ2FnZXMsIGZyYW1ld29ya3MsIGJpYmxpb3Row6hxdWVzLCBvdXRpbHMsIGxvZ2ljaWVscywgcGxhdGVmb3JtZXMgY2xvdWQsIGJhc2VzIGRlIGRvbm7DqWVzLCBvdXRpbHMgZGUgbW9uaXRvcmluZywgbcOpdGhvZGVzIChTY3J1bSwgVEREKSwgZG9tYWluZXMgdGVjaG5pcXVlcyAobWFjaGluZSBsZWFybmluZywgZGF0YSBtb2RlbGluZywgc8OpY3VyaXTDqSBhcHBsaWNhdGl2ZSkuCgoqKk5PTioqIDoKLSB0w6JjaGVzIG91IGFjdGl2aXTDqXMgKGTDqXBsb2llbWVudCwgdGVzdHMsIHJldnVlcyBkZSBjb2RlLCBpbnTDqWdyYXRpb25zIEFQSSwgcsOpZGFjdGlvbiBkZSBzcMOpY2lmaWNhdGlvbnMpIDsKLSBjb25jZXB0cyBnw6luw6lyaXF1ZXMgKEFQSSBzZXVsZSwgY2FjaGUsIGV2ZW50cywgcXVldWVzLCB3b3JrZXJzLCB2ZXJyb3VzLCBzY3JpcHRzLCBwbGFucywgcmVxdWlyZW1lbnRzLCBwaXBlbGluZXMgZGUgZG9ubsOpZXMsIGF1dGhlbnRpZmljYXRpb24pIDsKLSBzZWN0ZXVycyBkJ2FjdGl2aXTDqSAocmV0YWlsLCBDUEcsIGUtY29tbWVyY2UgY29tbWUgc2VjdGV1ciwgbWljcm9maW5hbmNlKSA7Ci0gYW5uw6llcyBkJ2V4cMOpcmllbmNlLCBkaXBsw7RtZXMsIHF1YWxpdMOpcyBodW1haW5lcyA7Ci0gbm9tcyBkJ2VudHJlcHJpc2Ugc2V1bHMgKMKrIE1pY3Jvc29mdCDCuywgwqsgdGVjaG5vbG9naWVzIE1pY3Jvc29mdCDCuykgOwotIGludGl0dWzDqXMgZGUgcsO0bGUgKMKrIEZ1bGwtU3RhY2sgwrspIGV0IG1vZMOobGVzIMOpY29ub21pcXVlcyAowqsgTWljcm8tU2FhUyDCuykuCgojIyMgUsOoZ2xlcwoKMS4gKipFeGhhdXN0aXZpdMOpKiogOiB0b3V0ZXMgbGVzIGNvbXDDqXRlbmNlcyBxdWkgcGFzc2VudCBsZSB0ZXN0LCB5IGNvbXByaXMgY2VsbGVzIGNpdMOpZXMgZW4gZXhlbXBsZSwgYXByw6hzIMKrIG91IMK7LCBlbnRyZSBwYXJlbnRow6hzZXMgb3UgZGFucyB1bmUgbGlzdGUgwqsgdW4gcGx1cyDCuyAoUHJvbWV0aGV1cywgR3JhZmFuYSwgTmFnaW9zLCBDZW50cmVvbiwgQ2xvdWRmbGFyZSwgTGV0J3MgRW5jcnlwdCwgUHVzaGVyLCBFTEsgU3RhY2vigKYpLgoyLiAqKlbDqXJpZmlhYmxlKiogOiBjaGFxdWUgw6l0aXF1ZXR0ZSBkb2l0IHBvdXZvaXIgw6p0cmUgcmV0cm91dsOpZSBkYW5zIGxlIHRleHRlIGRlIGwnb2ZmcmUuIFNldWxlcyBleGNlcHRpb25zIDogdW4gbm9tIGNhbm9uaXF1ZSAocsOoZ2xlcyAzIMOgIDUpIGV0IHVuZSBzw6lwYXJhdGlvbiAocsOoZ2xlIDYpLgozLiAqKkFscGhhYmV0IGxhdGluKiogcXVlbGxlIHF1ZSBzb2l0IGxhIGxhbmd1ZSBkZSBsJ29mZnJlIDogwqsgUHl0aG9uIMK7IGV0IG5vbiDCqyDYqNin2YrYq9mI2YYgwrssIMKrIEphdmFTY3JpcHQgwrsgZXQgbm9uIMKrINis2KfZgdinINiz2YPYsdmK2KjYqiDCuy4gVW5lIGNvbXDDqXRlbmNlIHNhbnMgw6lxdWl2YWxlbnQgdGVjaG5pcXVlIGNvbm51IGdhcmRlIHNvbiBub20gZCdvcmlnaW5lICjCqyDYs9mE2KkgwrssIMKrINmF2YbYtdipINiy2K8gwrspLgo0LiAqKkdyYXBoaWUgb2ZmaWNpZWxsZSoqIDogwqsgQXp1cmUgRGV2T3BzIMK7LCDCqyBKYXZhU2NyaXB0IMK7LCDCqyBQb3N0Z3JlU1FMIMK7LCDCqyBOb2RlLmpzIMK7LCDCqyBQZXJsIMK7LCDCqyBRbGlrVmlldyDCuy4gT24gcmV0aXJlIGxlcyBudW3DqXJvcyBkZSB2ZXJzaW9uICjCqyBKYXZhIDE3IMK7IOKGkiDCqyBKYXZhIMK7LCDCqyBQeXRob24zIMK7IOKGkiDCqyBQeXRob24gwrspLgo1LiAqKlVuIHNldWwgbm9tIHBhciBjb21ww6l0ZW5jZSBzdXIgdG91dCBsZSBqZXUqKiA6CgogICB8IEZvcm1lIHJldGVudWUgfCBWYXJpYW50ZXMgcmVtcGxhY8OpZXMgfAogICB8LS0tfC0tLXwKICAgfCBBUEkgUkVTVCB8IFJFU1QgQVBJLCBSRVNUZnVsIEFQSXMsIFJFU1RmdWwgc2VydmljZXMsIEFQSXMgUkVTVCwgUkVTVCBBUElzIHwKICAgfCBSZWFjdCB8IFJlYWN0LmpzIHwKICAgfCBBenVyZSB8IE1pY3Jvc29mdCBBenVyZSB8CiAgIHwgR0NQIHwgR29vZ2xlIENsb3VkIFBsYXRmb3JtIHwKICAgfCBTUUwgU2VydmVyIHwgTWljcm9zb2Z0IFNRTCBTZXJ2ZXIgfAogICB8IG1pY3Jvc2VydmljZXMgfCBtaWNyby1zZXJ2aWNlcywgTWljcm9zZXJ2aWNlcyB8CiAgIHwgcHJvZ3JhbW1hdGlvbiBvcmllbnTDqWUgb2JqZXQgfCBPT1AsIFBPTywgUHJvZ3JhbW1hdGlvbiBPcmllbnTDqWUgT2JqZXQgfAogICB8IGRhdGEgbW9kZWxpbmcgfCBtb2TDqWxpc2F0aW9uIGRlIGRvbm7DqWVzLCBkYXRhIG1vZGVsbGluZyB8CiAgIHwgZGF0YSB3YXJlaG91c2UgfCBkYXRhIHdhcmVob3VzaW5nIHwKICAgfCBkYXRhIGdvdmVybmFuY2UgfCBnb3V2ZXJuYW5jZSBkZSBsYSBkb25uw6llIHwKICAgfCBOTFAgfCB0cmFpdGVtZW50IGF1dG9tYXRpcXVlIGR1IGxhbmdhZ2UgbmF0dXJlbCB8CiAgIHwgYmFzZXMgdmVjdG9yaWVsbGVzIHwgdmVjdG9yIGRhdGFiYXNlcywgbW90ZXVycyB2ZWN0b3JpZWxzIHwKICAgfCBHRFBSIHwgUkdQRCB8CiAgIHwgU29mdHdhcmUgVGVzdGluZyB8IHNvZnR3YXJlIHRlc3RpbmcgbWV0aG9kb2xvZ2llcyB8CiAgIHwgc8OpY3VyaXTDqSBkZSBsJ2luZm9ybWF0aW9uIHwgaW5mb3JtYXRpb24gc2VjdXJpdHksINij2YXZhiDYp9mE2YXYudmE2YjZhdin2KogfAogICB8IHJlc3BvbnNpdmUgZGVzaWduIHwgUmVzcG9uc2l2ZSB8CiAgIHwgVUkgZGVzaWduLCBVWCBkZXNpZ24gfCBVSS9VWCwg2KrYtdmF2YrZhSBVSSBVWCB8CiAgIHwgRUxLIFN0YWNrIHwgRUxLIHwKCjYuICoqVG91am91cnMgc8OpcGFyZXIgbGVzIGNvbXDDqXRlbmNlcyBjb21wb3PDqWVzKiogOiDCqyBHaXQvR2l0TGFiIMK7IOKGkiDCqyBHaXQgwrssIMKrIEdpdExhYiDCuyA7IMKrIEhUTUwvQ1NTL0phdmFTY3JpcHQgwrsg4oaSIHRyb2lzIGVudHLDqWVzIDsgwqsgTm8tQ29kZSAvIExvdy1Db2RlIMK7IOKGkiDCqyBOby1Db2RlIMK7LCDCqyBMb3ctQ29kZSDCuyA7IMKrIEFnaWxlL1NjcnVtIMK7IOKGkiDCqyBBZ2lsZSDCuywgwqsgU2NydW0gwrsgOyDCqyBFVEwgLyBFTFQgwrsg4oaSIMKrIEVUTCDCuywgwqsgRUxUIMK7IDsgwqsgU1NMIC8gVExTIMK7LCDCqyBTU0ggLyBTRlRQIMK7LCDCqyBIVFRQIC8gSFRUUFMgwrssIMKrIExpbnV4L1VuaXggwrssIMKrIEFXUyAvIEVDMiDCuyBkZSBtw6ptZS4KICAgKkV4Y2VwdGlvbnMqIDogbGVzIG5vbXMgcXVpIGNvbnRpZW5uZW50IHVuIMKrIC8gwrsgbWFpcyBkw6lzaWduZW50IHVuZSBzZXVsZSBjaG9zZSByZXN0ZW50IGVudGllcnMgOiBDSS9DRCwgVENQL0lQLCBQTC9TUUwsIFB1Yi9TdWIsIHRlc3RzIEEvQi4KNy4gKipJQSoqIDogwqsgSUEgwrssIMKrIEFJIMK7IGV0IMKrIGludGVsbGlnZW5jZSBhcnRpZmljaWVsbGUgwrsg4oaSIHRvdWpvdXJzIMKrIGludGVsbGlnZW5jZSBhcnRpZmljaWVsbGUgwrsuIENvbXBvc8OpcyA6IMKrIElBIGfDqW7DqXJhdGl2ZSDCuyAvIMKrIEdlbmVyYXRpdmUgQUkgwrsg4oaSIMKrIGludGVsbGlnZW5jZSBhcnRpZmljaWVsbGUgZ8OpbsOpcmF0aXZlIMK7IDsgwqsgSUEgYWdlbnRpcXVlIMK7IC8gwqsgQWdlbnRpYyBBSSDCuyAvIMKrIEFJIEFnZW50cyDCuyDihpIgwqsgaW50ZWxsaWdlbmNlIGFydGlmaWNpZWxsZSBhZ2VudGlxdWUgwrsuCjguICoqQXVjdW4gZG91YmxvbioqIGRhbnMgdW5lIG3Dqm1lIG9mZnJlIChjb21wYXJhaXNvbiBpbnNlbnNpYmxlIMOgIGxhIGNhc3NlKS4KOS4gKipUZXN0cyoqIDogwqsgdGVzdHMgwrsgZXN0IHVuZSB0w6JjaGUgKE5PTikuIFBvdXIgdW4gcG9zdGUgUUEsIGxlIGRvbWFpbmUgcmVzdGUgdW5lIGNvbXDDqXRlbmNlIDogwqsgU29mdHdhcmUgVGVzdGluZyDCuywgwqsgYXV0b21hdGlzYXRpb24gZGVzIHRlc3RzIMK7LgoxMC4gKipTdGFjayBkZSBsJ2VudHJlcHJpc2UqKiA6IHVuZSB0ZWNobm9sb2dpZSBjaXTDqWUgc2V1bGVtZW50IGRhbnMgbGEgcHLDqXNlbnRhdGlvbiBkZSBsJ2VudHJlcHJpc2UgKMKrIG5vcyBzb2x1dGlvbnMgdXRpbGlzZW50IEFuZ3VsYXIsIEMj4oCmIMK7LCDCqyBCZXN0TGFiIGVzdCBzcMOpY2lhbGlzw6kgZW4gSUEsIERhdGEgU2NpZW5jZSDCuykgbidlc3QgcGFzIHVuZSBjb21ww6l0ZW5jZSBkZW1hbmTDqWUuIEVsbGUgbidlc3QgcGFzIGFubm90w6llLgoxMS4gKipSdWJyaXF1ZSDCqyDYp9mE2YXZh9in2LHYp9iqIMK7IGRlcyBvZmZyZXMgYXJhYmVzIChNb3N0YXFsKSoqIDogc2VzIMOpdGlxdWV0dGVzIHBhc3NlbnQgbGUgdGVzdCBldCBzb250IGNvbnZlcnRpZXMgZW4gbm9tcyBjYW5vbmlxdWVzIDoKICAgIC0g2KrYt9mI2YrYsSDYp9mE2KjYsdmF2KzZitin2Kog4oaSIGTDqXZlbG9wcGVtZW50IGxvZ2ljaWVsCiAgICAtINio2LHZhdis2Kkg2YXZiNin2YLYuSwg2KrYt9mI2YrYsSDYp9mE2YjZitioIOKGkiBkw6l2ZWxvcHBlbWVudCB3ZWIKICAgIC0g2YfZhtiv2LPYqSDYp9mE2KjYsdmF2KzZitin2Kog4oaSIGfDqW5pZSBsb2dpY2llbAogICAgLSDYqNix2YXYrNipINin2YTYqti32KjZitmC2KfYqiwg2KrYt9mI2YrYsSDYp9mE2KrYt9io2YrZgtin2Kog4oaSIGTDqXZlbG9wcGVtZW50IGQnYXBwbGljYXRpb25zCiAgICAtINiq2LXZhdmK2YUg2YXZiNin2YLYuSwg2KrYtdmF2YrZhSDZhdmI2YLYuSDYpdmE2YPYqtix2YjZhtmKIOKGkiB3ZWIgZGVzaWduCiAgICAtINio2LHZhdis2Kkg2KPZhtiv2LHZiNmK2K8g4oaSIEFuZHJvaWQgOyDYqNix2YXYrNipIGlPUywg2KjYsdmF2KzYqSDYqti32KjZitmC2KfYqiDYo9mK2YHZiNmGIOKGkiBpT1MKCiAgICBMZXMgw6l0aXF1ZXR0ZXMgcXVpIHNvbnQgZGVzIHTDomNoZXMgcmVzdGVudCBOT04gOiDYpdmG2LTYp9ihINmF2YjZgti5INil2YTZg9iq2LHZiNmG2YosINil2YbYtNin2KEg2YXYqtis2LEg2KXZhNmD2KrYsdmI2YbZiiwg2KrYtdmF2YrZhSDZhdiq2KzYsSDYpdmE2YPYqtix2YjZhtmKLCDYqti12YXZitmFINmF2YbZitmILCDYp9iz2KrZg9i02KfZgSDYp9mE2KPYrti32KfYoSDZiNil2LXZhNin2K3Zh9inLiBJbCBlbiB2YSBkZSBtw6ptZSBwb3VyIMKrINio2LHZhdis2Kkgwrsgc2V1bCwgdHJvcCBnw6luw6lyaXF1ZS4KCiMjIyA1IGV4ZW1wbGVzIE9VSQoKfCBFeHRyYWl0IGRlIGwnb2ZmcmUgfCDDiXRpcXVldHRlcyB8CnwtLS18LS0tfAp8IMKrIE1vbml0b3JpbmcgJiBPYnNlcnZhYmlsaXTDqSA6IFByb21ldGhldXMsIEdyYWZhbmEsIEVMSyBTdGFjaywgQ2VudHJlb24gb3UgTmFnaW9zIMK7IHwgUHJvbWV0aGV1cywgR3JhZmFuYSwgRUxLIFN0YWNrLCBDZW50cmVvbiwgTmFnaW9zLCBtb25pdG9yaW5nLCBvYnNlcnZhYmlsaXTDqSB8CnwgwqsgR2VzdGlvbiBkZXMgY2VydGlmaWNhdHMgKExldCdzIEVuY3J5cHQsIENsb3VkZmxhcmUsIEdhbmRpLi4uKSDCuyB8IExldCdzIEVuY3J5cHQsIENsb3VkZmxhcmUsIEdhbmRpIHwKfCDCqyBHaXQgLyBHaXRMYWIgwrsgOyDCqyBKZW5raW5zIG91IEdpdExhYiBDSSDCuyB8IEdpdCwgR2l0TGFiLCBKZW5raW5zLCBHaXRMYWIgQ0kgfAp8IMKrINin2YTZhdmH2KfYsdin2Ko6IEhUTUwgQ1NTINis2KfZgdinINiz2YPYsdmK2KjYqiBQSFAg2KjYp9mK2KvZiNmGIMK7IHwgSFRNTCwgQ1NTLCBKYXZhU2NyaXB0LCBQSFAsIFB5dGhvbiB8CnwgwqsgQ29tcMOpdGVuY2VzIGF2w6lyw6llcyBlbiBtb2TDqWxpc2F0aW9uIGRlIGRvbm7DqWVzIMK7IDsgwqsgTcOpdGhvZG9sb2dpZXMgQWdpbGUgKHNjcnVtKSDCuyB8IGRhdGEgbW9kZWxpbmcsIEFnaWxlLCBTY3J1bSB8CgojIyMgNSBleGVtcGxlcyBOT04KCnwgRXh0cmFpdCBkZSBsJ29mZnJlIHwgUG91cnF1b2kgY2Ugbidlc3QgcGFzIHVuZSBjb21ww6l0ZW5jZSB8CnwtLS18LS0tfAp8IMKrIE9wdGltaXNlciBsZXMgcGVyZm9ybWFuY2VzIDogcmVxdcOqdGVzLCBpbmRleCwgTisxLCBjYWNoZSwgd29ya2VycyDCuyB8IGNvbmNlcHRzIGfDqW7DqXJpcXVlcyB8CnwgwqsgRMOpdmVsb3BwZXIgZXQgaW50w6lncmVyIGRlcyBBUEkgwrsgOyDCqyBDb25jZXZvaXIgZGVzIEFQSSBldCBhc3N1cmVyIGwnaW50w6lncmF0aW9uIMK7IHwgwqsgQVBJIMK7IHNldWxlID0gY29uY2VwdCBnw6luw6lyaXF1ZSAobWFpcyDCqyBBUEkgUkVTVCDCuyA9IE9VSSkgfAp8IMKrIERlZXAgZXhwZXJ0aXNlIHdpdGggQ1BHIG9yIHJldGFpbCBkYXRhIMK7IHwgc2VjdGV1cnMgZCdhY3Rpdml0w6kgfAp8IMKrIFLDqWFsaXNlciBkZXMgcmV2dWVzIGRlIGNvZGUgZXQgZGUgc8OpY3VyaXTDqSDCuyA7IMKrIETDqWZpbmlyIGxlcyBwcm9jw6lkdXJlcyBkZSBkw6lwbG9pZW1lbnQgwrsgOyDCqyBSw6lkaWdlciBsZXMgcGxhbnMgdGVjaG5pcXVlcyDCuyB8IHTDomNoZXMgfAp8IMKrIDUgw6AgMTAgYW5zIGQnZXhww6lyaWVuY2UgwrsgOyDCqyBCYWMrNSBlbiBpbmZvcm1hdGlxdWUgwrsgOyDCqyBBdXRvbm9taWUsIHJpZ3VldXIgwrsgOyDCqyB0ZWNobm9sb2dpZXMgTWljcm9zb2Z0IMK7IHwgZXhww6lyaWVuY2UsIGRpcGzDtG1lLCBxdWFsaXTDqXMgaHVtYWluZXMsIG5vbSBkJ2VudHJlcHJpc2Ugc2V1bCB8Cg==').decode('utf-8')
RACINE = '/content/drive/MyDrive/job-offer-extractor'
trouves = glob.glob(f'{RACINE}/**/GUIDE_ANNOTATION.md', recursive=True)
GUIDE = trouves[0] if trouves else f'{RACINE}/GUIDE_ANNOTATION.md'
if os.path.exists(GUIDE):
    txt = open(GUIDE, encoding='utf-8').read()
    m = re.search(r'^##\s+Comp[ée]tences.*?(?=^##\s|\Z)', txt, flags=re.M | re.S)
    txt = txt[:m.start()] + SECTION + '\n' + txt[m.end():] if m else txt.rstrip() + '\n\n' + SECTION
    print('Section remplacée/ajoutée dans', GUIDE)
else:
    txt = '# Guide d’annotation\n\n' + SECTION
    print('Guide créé :', GUIDE)
open(GUIDE, 'w', encoding='utf-8').write(txt)
print(os.listdir(DOSSIER)); print(os.listdir(RESULTATS))